# Step 1: Install Required Libraries

In this step, we install all the libraries required for building our AI-powered Movie Recommendation Agent.

Libraries used:

- **Pandas** → Load and manipulate the movie dataset.
- **Sentence Transformers** → Convert movie descriptions into semantic embeddings.
- **ChromaDB** → Store embeddings in a vector database for semantic search.
- **Google Generative AI** → Generate natural-language explanations for recommendations using Gemini.

In [ ]:
# Install the required Python libraries

!pip install -q pandas sentence-transformers chromadb google-generativeai

# Step 2: Import Libraries

After installing the required packages, we import them into our notebook.

Each library serves a different purpose in building the recommendation system.

In [ ]:
# Data manipulation
import pandas as pd

# Working with ZIP files
import zipfile

# Access uploaded files in Google Colab
from google.colab import files

# Generate semantic embeddings
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# Google Gemini API
import google.generativeai as genai

# Operating system utilities
import os

# Step 3: Upload the Dataset

The IMDB movie dataset is stored as a ZIP file.

In this step, we upload the ZIP file from our computer into Google Colab.

In [ ]:
# Upload the dataset (.zip)

uploaded = files.upload()

# Step 4: Extract the Dataset

Since the dataset is compressed inside a ZIP file, we need to extract it before loading it into Pandas.

The extracted CSV file will be stored inside a folder named **data**.

In [ ]:
# Get the uploaded ZIP filename
zip_filename = list(uploaded.keys())[0]

# Extract all files from the ZIP into the "data" folder
with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall("data")

print("Dataset extracted successfully!")

# Step 5: Verify the Extracted Files

Before loading the dataset, we verify that the CSV file was extracted correctly.

In [ ]:
# Display all files inside the data folder

os.listdir("data")

# Step 6: Load the Dataset

Now we load the CSV file into a Pandas DataFrame.

A DataFrame is a table-like structure that allows us to manipulate and analyze data efficiently.

In [ ]:
# Load the movie dataset

movies = pd.read_csv("data/tmdb_5000_movies.csv")

# Display the first five rows

movies.head()

# Step 7: Exploratory Data Analysis (EDA)

Before building our AI model, it is important to understand the data.

We will:

- View the dataset structure.
- Check the data types.
- Identify missing values.
- Generate summary statistics.

In [ ]:
# Display the first five rows
movies.head()

In [ ]:
# Display information about the dataset
movies.info()

In [ ]:
# Generate descriptive statistics
movies.describe()

In [ ]:
# Count missing values in each column
movies.isnull().sum()

# Step 8: Data Cleaning

Machine learning models perform better when the dataset is clean.

In this step we will:

- Remove missing values.
- Remove duplicate movies.
- Reset the DataFrame index.

In [ ]:
# Remove rows with missing values
movies = movies.dropna()

# Remove duplicate rows
movies = movies.drop_duplicates()

# Reset the DataFrame index
movies.reset_index(drop=True, inplace=True)

print(f"Dataset contains {len(movies)} movies after cleaning.")

# Step 9: Create Movie Documents

Semantic search works best when each movie is represented as meaningful text.

We combine important columns such as:

- Title
- Genre
- Overview
- Director
- Stars

into a single text document for each movie.

In [ ]:
# Display all column names in the dataset
print(movies.columns.tolist())
# Combine the most useful text fields into one document
movies["document"] = (
    movies["title"].fillna("") + ". " +
    movies["genres"].fillna("") + ". " +
    movies["overview"].fillna("")
)

# Preview the generated documents
movies["document"].head()

# Step 10: Generate Semantic Embeddings

Embeddings convert text into numerical vectors.

These vectors capture the semantic meaning of each movie, allowing us to search based on meaning rather than exact keywords.

In [ ]:
# Load the Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for each movie document
embeddings = model.encode(
    movies["document"].tolist(),
    show_progress_bar=True
)

print(f"Generated {len(embeddings)} embeddings.")

# Step 11: Create a Vector Database

ChromaDB stores the movie embeddings.

Instead of searching through every movie manually, we can retrieve the most semantically similar movies almost instantly.

In [ ]:
# Create a persistent ChromaDB client
client = chromadb.PersistentClient(path="chroma_db")

# Create (or get) the movie collection
collection = client.get_or_create_collection(
    name="movies"
)

# Step 12: Store Embeddings

Each movie is stored together with:

- its embedding
- its title
- additional metadata

This allows semantic retrieval later.

In [ ]:
# Store the movies in ChromaDB

collection.add(
    ids=[str(i) for i in movies.index],
    documents=movies["document"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "title": row["title"],
            "genre": row["genres"],
            "rating": float(row["vote_average"])
        }
        for _, row in movies.iterrows()
    ]
)

print("✅ Movies successfully stored in ChromaDB!")

# Step 13: Semantic Search

The user enters a natural language query.

The embedding model converts the query into a vector, and ChromaDB retrieves the most relevant movies based on semantic similarity.

In [ ]:
# User query
query = "I want an emotional science fiction movie with space travel."

# Convert the query into an embedding
query_embedding = model.encode(query).tolist()

# Retrieve the top 5 most similar movies
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Display the retrieved movie metadata
results["metadatas"][0]

# Step 14: Configure the Gemini API

After retrieving the most relevant movies using semantic search, we use Google's Gemini model to explain **why** each movie matches the user's request.

This is the **Generation** part of the Retrieval-Augmented Generation (RAG) pipeline.

Workflow:

User Query
        ↓
Semantic Search (ChromaDB)
        ↓
Top Matching Movies
        ↓
Gemini
        ↓
Natural Language Explanation

In [ ]:
# Import the Gemini library
import google.generativeai as genai

# Prompt the user to enter their Gemini API key
api_key = input("Enter your Gemini API Key: ")

# Configure Gemini with the provided API key
genai.configure(api_key=api_key)

# Initialize the Gemini model
model_gemini = genai.GenerativeModel("gemini-2.5-flash")

print("✅ Gemini configured successfully!")

In [ ]:
response = model_gemini.generate_content("Hello! Introduce yourself in one sentence.")

print(response.text)

# Step 15: Create the Recommendation Function

This function performs three tasks:

1. Converts the user's query into an embedding.
2. Retrieves the most relevant movies from ChromaDB.
3. Uses Gemini to explain why each movie is recommended.

In [ ]:
def recommend_movie(user_query, top_k=5):
    """
    Retrieves the most relevant movies using semantic search
    and generates personalized explanations using Gemini.
    """

    # Convert the user's query into an embedding
    query_embedding = model.encode(user_query).tolist()

    # Retrieve the most similar movies
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    # Extract movie metadata
    recommendations = results["metadatas"][0]

    # Build a readable movie list
    movie_list = ""

    for movie in recommendations:
        movie_list += (
            f"- {movie['title']} "
            f"(Genre: {movie['genre']}, "
            f"Rating: {movie['rating']})\n"
        )

    # Create a prompt for Gemini
    prompt = f"""
You are an AI movie recommendation assistant.

A user asked:

"{user_query}"

The semantic search system retrieved these movies:

{movie_list}

Explain why each movie matches the user's request.
Keep the explanation friendly and concise.
"""

    # Generate Gemini response
    response = model_gemini.generate_content(prompt)

    return response.text

# Step 16: Test the Movie Recommendation Agent

Let's test the complete Retrieval-Augmented Generation (RAG) pipeline.

The user enters a natural language request.

The system:
- converts the request into an embedding,
- retrieves similar movies using ChromaDB,
- asks Gemini to explain the recommendations.

In [ ]:
recommend_movie(
    "I want an emotional science fiction movie with space travel."
)

In [ ]:
recommend_movie(
    "I want an action movie with superheroes."
)


# Step 17: Persist the ChromaDB Database

Since we created a persistent ChromaDB client, the embeddings are automatically saved to disk.

This means we don't need to regenerate embeddings every time we restart the notebook.

In [ ]:
print("✅ Vector database saved successfully.")